<h1><center>Laboratorio 6: El Pudú no pierde el rastro 🦌</center></h1>

<center><strong>IA7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

<div style="text-align: center; margin: 16px 0;">
  <img src="assets/don_pudencio.png" alt="Don Pudencio, mascota pudú de Pudubella con chaleco verde y credencial de Gerente de Tienda, saludando frente a la tienda Pudubella en el Mall Cordillera." style="width: 420px; max-width: 100%; height: auto;">
</div>


---

### Cuerpo Docente

- Profesores: Pablo Badilla e Ignacio Núñez
- Auxiliar: Sofía Chávez
- Ayudantes: Javiera Arévalo, Tamara Cornejo, Ignacio Reyes

### Equipo

- Nombre de estudiante 1:
- Nombre de estudiante 2:

### Link de la Pull Request

Peguen la URL HTTPS completa de la Pull Request:

`https://github.com/<organizacion>/<repositorio>/pull/<numero>`

---

### Reglas

- Trabajar en equipos de hasta dos personas.
- Mantener las transformaciones de datos en Polars. DuckDB y DuckLake son la capa de persistencia local.
- No modificar los Parquet de `data/raw/` ni los de `data/raw_invalid/`.
- No usar `pandas` ni ciclos `for` o `while` para transformar filas. Las excepciones son la infraestructura provista y la lectura de las cuatro fuentes.

### Flujo de trabajo

El notebook sirve para inspeccionar evidencia y tomar decisiones. Las funciones reutilizables viven bajo `src/pudulake/`, los contratos viven en `contracts/` y las pruebas entregadas indican cuándo una etapa está cerrada.

Ejecutar al terminar cada etapa:

```bash
uv run pytest -m etapa1
uv run pytest -m etapa2
uv run pytest -m etapa3
uv run pytest -m etapa4
```

### Configuración del Ambiente

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137,87,229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Preparar el ambiente local</strong>
  Abran una terminal en la raíz de este laboratorio. <code>uv sync --locked --all-groups</code> instala las versiones fijadas de Polars, DuckDB, Prefect y las herramientas de desarrollo. DuckLake es una extensión de DuckDB: instálenla y comprueben que carga antes de la sesión. En una segunda terminal inicien Prefect Server y manténganlo abierto.
</div>

```bash
uv sync --locked --all-groups
uv run python -c 'import duckdb; con = duckdb.connect(); con.execute("INSTALL ducklake"); con.execute("LOAD ducklake")'
uv run prefect server start
```

El servidor queda disponible en <code>http://127.0.0.1:4200</code>. En la terminal del laboratorio, ejecuten <code>uv run python main.py run</code>. Si la extensión o el servidor no inicia, deténganse y registren el mensaje: no continúen suponiendo que la corrida quedó observada.

## Contexto

<!-- LORE -->

<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📜 El encargo de Don Pudencio</strong>

Pudubella lleva años vendiendo productos a lo largo de Chile. Sus tiendas reciben pedidos, procesan pagos, despachan productos y acumulan registros de cada una de estas operaciones. Datos hay de sobra. El problema es que, hasta ahora, nadie los ha convertido en información que permita entender realmente qué está ocurriendo en el negocio.

Don Pudencio, gerente de una de sus principales tiendas, quiere priorizar una campaña para clientes que probablemente recomprarán durante los próximos 30 días. Para eso necesita una estimación de <strong>propensión a recompra</strong>: la probabilidad de que una persona vuelva a comprar dentro de una ventana futura. Pronto descubre que no basta con sumar columnas. Los datos provienen de distintas fuentes, tienen diferentes unidades de observación y no todas las relaciones entre ellas son evidentes.

Antes de estimar esa propensión, Pudubella necesita datos confiables. Una fecha mal interpretada, un pago negativo, una orden sin pago o una unión con la cardinalidad equivocada cambian Recency, Frequency o Monetary y entregan al modelo una señal falsa. Don Pudencio prefiere descubrir esas fallas antes de usar el resultado para decidir a quién contactar.

En este laboratorio tomarán el histórico público de Olist como representación de los datos de Pudubella y construirán un lakehouse local que permita llevarlos desde su estado original hasta productos analíticos confiables. El problema secundario es construir RFM: será la tabla de variables reproducible que un modelo de propensión a recompra podrá recibir después. En este laboratorio no entrenarán ese modelo. El desafío es poder explicar **de dónde vienen los datos, qué reglas cumplen y qué partes del sistema dependen de ellos**.

</div>


## Objetivos

- Materializar fuentes Parquet en una arquitectura Bronze → Silver → Gold dentro de un lakehouse local.
- Usar contratos YAML y validaciones para decidir cuándo una tabla es confiable.
- Construir ventas diarias y variables RFM con semántica explícita para estimar propensión a recompra en un modelo posterior.
- Orquestar y observar el pipeline con Prefect, DuckLake y una CLI local.

### Evaluación

| Etapa | Evidencia | Puntaje |
| --- | --- | ---: |
| 1 | Lectura 0,2; EDA, llaves y cardinalidad 0,4; Bronze modularizado 0,4 | 1,0 |
| 2 | Justificación 0,4; fechas y montos 0,6; relaciones 0,3; contratos y modularización Silver 0,9 | 2,2 |
| 3 | Cobertura y exclusiones 0,4; ventas diarias 0,3; RFM y segmentos 0,6; contratos y modularización Gold 0,5 | 1,8 |
| 4 | Corrida válida y snapshots 0,3; falla, preservación y lineage 0,5; evidencia de entrega 0,2 | 1,0 |
| **Total** | | **6,0** |

La nota se calcula como 1,0 + puntaje obtenido.

## Teoría

<!-- DEFINICIÓN -->

<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Ingeniería de datos</strong>
La <strong>ingeniería de datos</strong> diseña y opera los sistemas que reciben, almacenan, transforman y entregan datos. Su trabajo convierte fuentes dispersas en tablas que otras personas y sistemas pueden usar con una estructura, calidad y procedencia conocidas. En este laboratorio, esa base permite que las variables RFM puedan reutilizarse más adelante en un modelo sin tener que reconstruir su origen en cada corrida.

</div>

<!-- DEFINICIÓN -->

<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Data Lake</strong>
Un <strong>Data Lake</strong> almacena datos de muchas fuentes y conserva sus archivos cerca del formato en que llegaron. Eso permite volver al origen y aplicar transformaciones distintas según la necesidad posterior. Por sí solo, un lake no garantiza que una tabla publicada tenga esquema, calidad, versión o relaciones verificadas.

</div>


<img src="assets/de_process.png" alt="Diagrama de la arquitectura Medallion de Pudubella: fuentes de datos pasan por extraer, transformar y cargar hacia las capas Bronze, Silver y Gold de un lakehouse; los productos resultantes sirven para reportes, análisis y ciencia de datos." style="max-width:100%; height:auto;">

<!-- DEFINICIÓN -->

<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Lakehouse</strong>
Un <strong>lakehouse</strong> conserva la flexibilidad de un Data Lake y agrega mecanismos para publicar tablas de manera controlada: catálogo, esquemas, versiones y transacciones. Así, una tabla Gold puede servir tanto para un tablero como para preparar variables de un modelo predictivo sin perder el vínculo con las fuentes que la originaron.

</div>


<!-- DEFINICIÓN -->

<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 ETL: Extract, Transform, Load</strong> <strong>ETL</strong> organiza el paso desde una fuente hasta una salida utilizable. <strong>Extract</strong> lee datos desde archivos, bases de datos u otros sistemas; <strong>Transform</strong> tipa, relaciona, valida o agrega según una regla; y <strong>Load</strong> publica el resultado en su destino. Aquí, Bronze extrae y conserva, Silver transforma y valida, y Gold carga productos analíticos como ventas diarias y variables RFM.

</div>


<!-- DEFINICIÓN -->

<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Pipeline de datos</strong>
Un <strong>pipeline de datos</strong> es la secuencia reproducible de tareas y dependencias que lleva datos desde una fuente hasta un producto. Cada tarea recibe insumos, aplica una operación y deja una salida que otras tareas pueden necesitar. En este laboratorio, el pipeline mueve los Parquet por Bronze, Silver y Gold; si una validación de Silver falla, los productos Gold dependientes no se actualizan y las variables RFM no se usan como insumo de un modelo posterior.

</div>


<!-- DEFINICIÓN -->

<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Propensión a recompra</strong>
La <strong>propensión a recompra</strong> estima la probabilidad de que una persona vuelva a comprar durante una ventana futura, por ejemplo, los próximos 30 días. Un modelo posterior recibirá variables calculadas hasta una fecha de referencia —como RFM— y comparará esa información con si ocurrió o no una recompra después de esa fecha. Este laboratorio construye la base limpia y trazable de esas variables; no entrena ni evalúa el modelo.

</div>


---
## Etapa 1 — Bronze y procedencia (1,0 puntos)

<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📜 El primer corte</strong>
Don Pudencio recibió cuatro archivos y una promesa: “aquí está toda la operación”. No hay diccionario de datos ni alguien que pueda asegurar qué representa cada fila. Antes de preparar variables para estimar recompra, el equipo debe inventariar las fuentes y conservar una copia que permita volver al origen si aparece una contradicción.

</div>


<!-- DEFINICIÓN -->

<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

<strong style="display:block; margin-bottom:8px;">📖 Bronze</strong> <strong>Bronze</strong> es la capa que registra cada fuente tal como llegó, con solo los metadatos necesarios para identificar su origen o su carga. No corrige estados, fechas, montos ni relaciones de negocio. Esa copia es el punto de partida para reconstruir Silver y Gold, y permite verificar de qué datos provino una variable RFM si un modelo posterior entrega una predicción inesperada.

</div>


### 1.1 — Rutas y librerías

In [ ]:
from pathlib import Path
import sys

import polars as pl

RUTA_LAB = Path.cwd().parent
if str(RUTA_LAB) not in sys.path:
    sys.path.insert(0, str(RUTA_LAB))

RAW_DIR = Path("../data/raw")

### 1.2 — Lectura de las cuatro fuentes (0,2 puntos)

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 I/O, en breve</strong>
  I/O (<em>Input/Output</em>) es el intercambio entre un programa y un sistema externo. Leer un archivo es una entrada; escribir una tabla o un reporte es una salida. La lectura puede fallar porque el archivo no existe, está dañado o no tiene el formato esperado, por lo que el pipeline debe informar qué fuente no pudo recibir.
</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Parquet</strong>
  <strong>Parquet</strong> es un formato de archivos columnar: guarda los valores de cada columna juntos e incluye información sobre su esquema. Es adecuado para tablas porque permite leer solo las columnas necesarias y conserva tipos como fechas, números y texto. En Bronze se lee cada archivo sin renombrar ni transformar sus columnas.
</div>

Lean los cuatro archivos directamente para inspeccionarlos. Guarden las tablas
en `orders_raw`, `customers_raw`, `items_raw` y `payments_raw`. En esta primera
lectura no agreguen columnas ni cambien nombres o tipos: esas decisiones se
tomarán después de observar la evidencia.

In [ ]:
# Su código aquí
raise NotImplementedError(
    "Lean las cuatro fuentes Parquet antes de continuar con el EDA."
)

### 1.3 — Auditoría y claves (0,4 puntos)

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Análisis Exploratorio de Datos (EDA)</strong>

  El <strong>análisis exploratorio de datos</strong> (<em>Exploratory Data Analysis</em>, EDA) reúne evidencia antes de construir transformaciones, indicadores o variables para un modelo. Permite saber qué representa cada fuente, cómo está estructurada y qué problemas pueden cambiar la interpretación de sus resultados. No corrige todavía los datos: fundamenta las reglas que después aplicarán Silver y Gold.

  Un EDA debería revisar, al menos:

  <ul style="margin:10px 0 0 20px;">
    <li><strong>Dimensiones:</strong> número de filas y columnas.</li>
    <li><strong>Esquema:</strong> nombres de columnas y tipos de datos.</li>
    <li><strong>Unidad de observación:</strong> qué representa una fila.</li>
    <li><strong>Claves:</strong> columnas que identifican registros y posibles duplicados.</li>
    <li><strong>Valores ausentes:</strong> cantidad y distribución de nulos.</li>
    <li><strong>Distribuciones:</strong> rangos, valores frecuentes y posibles valores atípicos.</li>
    <li><strong>Categorías:</strong> valores posibles en variables discretas, como estados o tipos.</li>
    <li><strong>Relaciones entre fuentes:</strong> claves compartidas, cardinalidades y posibles registros huérfanos.</li>
  </ul>

  <p style="margin-top:10px;">
    El resultado del EDA debe permitir proponer contratos, anticipar qué uniones pueden multiplicar filas y definir qué registros entrarán a las variables RFM.
  </p>

</div>

A continuación, identifiquen qué contiene cada fuente de datos. Usen la función
`generar_eda` para hacerlo.

In [ ]:
from IPython.display import display
import polars as pl


def generar_eda(
    tabla: pl.DataFrame,
    nombre_fuente: str,
    n_muestra: int = 8,
    seed: int = 7202,
) -> None:
    """Muestra una exploración inicial de la fuente."""

    print(f"EDA rápido {nombre_fuente}\n")
    print("-" * 80)
    print(f"Muestra aleatoria - Semilla = {seed}")
    print("-" * 80)
    display(
        tabla.sample(
            n=min(n_muestra, tabla.height),
            seed=seed,
            shuffle=True,
        )
    )

    print("\n" + "-" * 80)
    print("Perfilamiento por columnas")
    print("-" * 80)

    nulos = tabla.null_count().transpose(
        include_header=True,
        header_name="columna",
        column_names=["nulos"],
    )

    distintos = tabla.select(pl.all().n_unique()).transpose(
        include_header=True,
        header_name="columna",
        column_names=["valores_distintos"],
    )

    esquema = pl.DataFrame(
        {
            "columna": tabla.columns,
            "tipo": [str(tipo) for tipo in tabla.schema.values()],
        }
    )

    if tabla.is_empty():
        porcentaje_nulos = pl.lit(None, dtype=pl.Float64)
    else:
        porcentaje_nulos = (pl.col("nulos") / tabla.height * 100).round(2)

    display(
        esquema.join(nulos, on="columna")
        .join(distintos, on="columna")
        .with_columns(
            porcentaje_nulos.alias("porcentaje_nulos")
        )
        .sort("porcentaje_nulos", descending=True)
    )

    print("\n\n" + "-" * 80)
    print("Estadísticas descriptivas")
    print("-" * 80)
    display(tabla.describe())

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Grano y cardinalidad</strong>
  El <strong>grano</strong> declara qué representa una fila: una orden, un ítem de una orden, un pago o una persona. La <strong>cardinalidad</strong> describe cuántas filas de una tabla pueden corresponder a una fila de otra: uno a uno, uno a muchos o muchos a muchos. Antes de hacer un <em>join</em>, comparen ambos granos; una relación muchos a muchos puede multiplicar filas y duplicar montos.
</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Llave</strong>
  Una <strong>llave</strong> es el conjunto mínimo de columnas que identifica de manera única cada registro de una tabla. Una tabla puede tener más de una llave candidata. Comprobarla implica buscar duplicados y nulos antes de usarla para unir fuentes o para contar compras por persona.
</div>

In [ ]:
generar_eda(orders_raw, "orders", n_muestra=10, seed=7202)

In [ ]:
generar_eda(customers_raw, "customers", n_muestra=10, seed=7202)

In [ ]:
generar_eda(items_raw, "items", n_muestra=10, seed=7202)

In [ ]:
generar_eda(payments_raw, "payments", n_muestra=10, seed=7202)

Registren ahora sus decisiones en una tabla Markdown. Para cada fuente indiquen
el grano, la llave candidata, la evidencia que la respalda (`n_unique`, nulos o
duplicados), la llave con que se relaciona con `orders` y su cardinalidad
esperada. Si no encuentran una llave candidata, declárenlo y expliquen qué
evidencia faltaría para aceptarla como llave.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Preguntas</strong>
  1. Completen la tabla de catálogo solicitada arriba y citen la evidencia que respalda cada llave candidata y cardinalidad.
  2. ¿Qué conteo anticipan que cambia si agregan pagos a nivel de ítem y por qué `customer_id` no basta para medir frecuencia de compra?
</div>

<code>Escriban sus respuestas aquí:</code>

### 1.4 — Conservación en Bronze (0,4 puntos)

<!-- MINI PROYECTO -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240,136,62,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🛠️ Modularización — Ingesta Bronze</strong>
  Ya verificaron en el notebook cómo leer las cuatro fuentes. Trasladen ahora esa lectura a `read_sources(raw_dir: Path)` en `src/pudulake/bronze.py`. La función recibe el directorio `raw_dir` y debe devolver un diccionario de tipo `dict[str, pl.DataFrame]` con las entradas `orders`, `customers`, `order_items` y `payments`, leídas desde sus respectivos archivos Parquet.

  La función no debe depender de las variables del notebook. Bronze conserva la fuente recibida: no renombren, eliminen, reordenen ni transformen columnas. Si falta cualquiera de los cuatro archivos, debe levantar `FileNotFoundError` e incluir el nombre de la fuente ausente en el mensaje.
</div>

In [ ]:
from src.pudulake.bronze import read_sources

# Después de modularizar, esta comprobación debe ejecutarse sin modificarla.
sources_bronze = read_sources(RAW_DIR)
pl.DataFrame(
    {
        "fuente": sorted(sources_bronze),
        "filas": [sources_bronze[name].height for name in sorted(sources_bronze)],
    }
)

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  Ejecutar `uv run pytest -m etapa1`.

  Las pruebas en `tests/test_bronze.py` comprueban que `read_sources` devuelve las cuatro fuentes sin alterar sus tablas y que, al faltar `payments.parquet`, informa la ausencia mediante `FileNotFoundError`.

</div>

---
## Etapa 2 — Silver, contratos y calidad (2,2 puntos)

<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 Don Pudencio necesita garantías</strong>
  Las fuentes llegaron, pero Don Pudencio no aceptará que una tabla se llame confiable solo porque se pudo leer. Un pago negativo, una clave huérfana o una fecha imposible alterarían las variables RFM y, después, la propensión estimada. Antes de usar una tabla, necesita saber qué representa una fila y qué regla impide publicar un dato que contradice su contrato.
</div>

### 2.1 — Predicción y justificación (0,4 puntos)

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Contrato de datos</strong>
  Un <strong>contrato de datos</strong> es un acuerdo explícito sobre lo que una tabla publica y las condiciones que debe cumplir para poder usarse. Puede declarar propósito, grano, llave, procedencia, tipos, política de nulos y relaciones con otras tablas.

  El contrato no corrige datos, sino que define qué se espera de ellos. En este laboratorio las fichas YAML declaran estructura y restricciones por tabla; `validate_relationships` comprueba por separado las relaciones entre fuentes. Por ejemplo, `payments` exige que `payment_value` sea finito y no negativo, y su `order_id` se verifica contra órdenes. Si una regla crítica falla, Silver no se publica; así RFM y un modelo posterior no reciben una señal basada en datos contradictorios.
</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Estándar y esquema local de contratos</strong>
  No existe un único formato obligatorio para todos los contratos de datos. Un referente abierto es <strong>Open Data Contract Standard</strong> (ODCS), que usa YAML para describir estructura, semántica, calidad, responsables y dónde vive un conjunto de datos. Su alcance es mayor que el de este laboratorio.

  Aquí usamos un esquema local, más pequeño y ejecutable. La fuente de verdad no es una plantilla externa: es `src/pudulake/contracts.py`. `REQUIRED_FIELDS` define los campos que deben aparecer en cada ficha; `ALLOWED_TYPES` fija el vocabulario de tipos; y `ALLOWED_RULES` define las restricciones permitidas. Completen los YAML a partir de esa interfaz y de la evidencia del EDA, no copiando un contrato genérico. En un proyecto real, el equipo puede adoptar ODCS completo o mantener un esquema propio si cubre las garantías que necesita validar.

  Este ejemplo, sobre préstamos de biblioteca, muestra la estructura mínima de una ficha. No corresponde a una tabla de Pudubella: úselo para reconocer la función de cada campo y definan los valores de sus contratos a partir del grano y las reglas de sus propias fuentes.

```yaml
table: silver.loans
purpose: Registrar préstamos con fecha de inicio interpretable.
grain: Una fila por loan_id.
primary_key: [loan_id]
owner: Biblioteca Central
lineage: [bronze.loans]
classification: interno
required_columns: [loan_id, member_id, started_at]
column_types:
  loan_id: string
  member_id: string
  started_at: datetime
allow_empty: false
constraints:
  loan_id:
    nullable: false
    unique: true
  member_id:
    nullable: false
  started_at:
    nullable: false
```

</div>

In [ ]:
orders_raw.select(
    pl.len().alias("filas"),
    pl.col("order_id").n_unique().alias("ordenes_distintas"),
    pl.col("order_status").value_counts().alias("estados"),
)

Antes de publicar una tabla en Silver, cada regla del contrato debe producir una decisión observable:

- Una ausencia (i.e., valor nulo) permitida se podría conservar y quedar registrada como tal
- Un valor que contradice el tipo o el rango esperado en su columna comúnmente se rechaza
- Y una relación entre fuentes se verifica mediante sus llaves.

Por ejemplo, `order_delivered_customer_date` puede ser nula si una orden aún no fue entregada. En cambio, el texto `"fecha_invalida"` no representa una fecha: convertirlo silenciosamente en nulo confundiría una falla de formato con una ausencia real.

Silver debe distinguir ambos casos antes de construir RFM, porque cada uno cambia la evidencia disponible para estimar recompra.

In [ ]:
fechas_biblioteca = pl.DataFrame(
    {"prestamo": [
        "2026-03-01 09:00:00",
        "fecha_invalida",
        None,
]}
)
fechas_biblioteca.with_columns(
    pl.col("prestamo")
    .str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False)
    .alias("prestamo_parseado")
)

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Joins en Polars</strong>
  Un <em>join</em> relaciona filas por el valor de una llave, no por su posición. En Polars, la tabla sobre la que se llama <code>.join()</code> es la tabla izquierda: <code>izquierda.join(derecha, on="order_id", how="...")</code>.

  Sean \(A\) y \(B\) los conjuntos de llaves de las tablas izquierda y derecha. La cardinalidad importa: unir una orden con sus ítems es uno a muchos; si ambas tablas tienen varias filas por la misma llave, el resultado puede multiplicar filas y montos.

  <ul style="margin:10px 0 0 20px;">
    <li><code>how="inner"</code>: conserva las llaves de \(A \cap B\), presentes en ambas tablas. Es el valor por defecto.</li>
    <li><code>how="left"</code>: conserva todas las llaves de \(A\). Si una llave pertenece a \(A \setminus B\), las columnas de la derecha quedan nulas.</li>
    <li><code>how="right"</code>: conserva todas las llaves de \(B\). Si una llave pertenece a \(B \setminus A\), las columnas de la izquierda quedan nulas.</li>
    <li><code>how="full"</code>: conserva todas las llaves de \(A \cup B\). Las columnas del lado sin correspondencia quedan nulas.</li>
    <li><code>how="semi"</code>: conserva las filas de la izquierda con llaves en \(A \cap B\), pero devuelve solo columnas de la izquierda.</li>
    <li><code>how="anti"</code>: conserva las filas de la izquierda con llaves en \(A \setminus B\), pero devuelve solo columnas de la izquierda. Aquí sirve para detectar pagos cuya orden no existe y órdenes entregadas sin pago.</li>
  </ul>
</div>

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Predicción — decisiones de Silver</strong>
  1. En el ejemplo, distingan la ausencia de la falla de conversión. ¿Por qué ambas no pueden terminar como un nulo?
  2. Identifiquen la llave de cada tabla Silver y la regla crítica de `payments`. ¿Qué producto Gold dejaría de ser confiable si esa regla falla?
  3. Si un pago referencia una orden inexistente, ¿qué devolvería el anti join desde pagos hacia órdenes y por qué debe detenerse Silver?
</div>

<code>Escriban su respuesta aquí:</code>

### 2.2 — Construcción exploratoria de Silver (1,8 puntos)

En esta sección implementarán las reglas primero sobre las tablas del notebook.
La meta es que puedan observar qué entra a Silver, qué se rechaza y por qué.
Cuando las tres comprobaciones funcionen, trasladarán exactamente esas decisiones
a funciones y contratos reutilizables al final de la etapa.

#### 2.2.1 — Fechas, estados y clientes

Construyan `orders_silver` desde `orders_raw` y `customers_silver` desde
`customers_raw`. No modifiquen las tablas Bronze. En `orders_silver`:

- conviertan a `Datetime` `order_purchase_timestamp`, `order_approved_at`,
  `order_delivered_carrier_date`, `order_delivered_customer_date` y
  `order_estimated_delivery_date`;
- conserven los nulos originales; si un valor no nulo no se puede convertir,
  levanten `ContractViolation`;
- acepten solo `approved`, `canceled`, `created`, `delivered`, `invoiced`,
  `processing`, `shipped` y `unavailable`;
- agreguen `delivery_timestamp_missing`, verdadero únicamente para una orden
  `delivered` cuya fecha de entrega al cliente sea nula;
- rechacen una orden entregada cuya fecha de entrega sea anterior a la compra.

`customers_silver` conserva `customer_id` y `customer_unique_id` sin agrupar:
una persona puede aparecer mediante varios `customer_id`, y esa consolidación
corresponde a RFM, no a Silver.

In [ ]:
# Su código aquí
raise NotImplementedError(
    "Construyan y verifiquen orders_silver y customers_silver en el notebook."
)

#### 2.2.2 — Montos admisibles

Construyan `items_silver` y `payments_silver`. Antes de conservar las tablas,
busquen precios, fletes y pagos negativos, `NaN` o infinitos. Si aparece uno,
levanten `ContractViolation`; no lo reemplacen por cero ni eliminen la fila.
Al terminar, muestren el mínimo de `price`, `freight_value` y `payment_value`.

In [ ]:
# Su código aquí
raise NotImplementedError(
    "Verifiquen los montos antes de construir items_silver y payments_silver."
)

#### 2.2.3 — Relaciones que permiten publicar Silver

Auditen las tres relaciones con un `anti join`: `orders.customer_id` contra
`customers.customer_id`, `order_items.order_id` contra `orders.order_id` y
`payments.order_id` contra `orders.order_id`. Construyan una tabla con el
número de filas huérfanas de cada relación. Si alguno es mayor que cero,
levanten `ContractViolation` e indiquen la relación fallida.

In [ ]:
# Su código aquí
raise NotImplementedError(
    "Auditen las tres relaciones Silver antes de continuar."
)

<!-- MINI PROYECTO -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240,136,62,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🛠️ Modularización — Entidades confiables</strong>
  Las reglas ya funcionan sobre las tablas del notebook. Trasládenlas ahora a `src/pudulake/silver.py`, sin escribir archivos ni publicar tablas desde esas funciones. Completen `build_orders`, `build_customers`, `build_order_items`, `build_payments` y `validate_relationships` con las mismas precondiciones y errores que verificaron arriba.

  Completen además las cuatro fichas Silver de `contracts/`: `silver_orders.yaml`, `silver_customers.yaml`, `silver_order_items.yaml` y `silver_payments.yaml`. Cada una debe declarar `table`, `purpose`, `grain`, `primary_key`, `owner`, `lineage`, `classification`, `required_columns`, `column_types`, `allow_empty` y `constraints`. Consulten `src/pudulake/contracts.py` para los tipos y reglas admitidos. Las funciones `build_*` aplican reglas semánticas; el flow usa `validate_contract` para comprobar las restricciones declaradas, como unicidad, nulos y rangos.

  Una fecha no interpretable, un monto negativo o no finito y una relación huérfana deben levantar `ContractViolation`. No oculten ese error dentro de un `try`: el flow debe detenerse antes de publicar Silver.
</div>

In [ ]:
from src.pudulake.contracts import load_contract, validate_contract
from src.pudulake.silver import (
    build_customers,
    build_order_items,
    build_orders,
    build_payments,
    validate_relationships,
)

# Después de modularizar, esta comprobación debe ejecutarse sin modificarla.
contracts_silver = {
    path.stem: load_contract(path)
    for path in sorted((RUTA_LAB / "contracts").glob("silver_*.yaml"))
}
orders_silver = build_orders(orders_raw)
customers_silver = build_customers(customers_raw)
items_silver = build_order_items(items_raw)
payments_silver = build_payments(payments_raw)
validate_contract(orders_silver, contracts_silver["silver_orders"])
validate_contract(customers_silver, contracts_silver["silver_customers"])
validate_contract(items_silver, contracts_silver["silver_order_items"])
validate_contract(payments_silver, contracts_silver["silver_payments"])
validate_relationships(
    orders_silver,
    customers_silver,
    items_silver,
    payments_silver,
)
print("Las cuatro entidades y las tres relaciones Silver son válidas.")

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  `uv run pytest -m etapa2` pasa.

  Las pruebas verifican que las fechas de órdenes queden tipadas, que una ausencia permitida no se confunda con una fecha inválida, que montos negativos o no finitos sean rechazados, que una clave foránea huérfana detenga la publicación y que las cuatro fichas Silver declaren la gobernanza requerida.
</div>

---
## Etapa 3 — Productos Gold y RFM (1,8 puntos)

<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 La decisión comercial</strong>
  Con las entidades ya verificadas, Don Pudencio necesita dos productos distintos. Operaciones necesita ventas por día; fidelización necesita estimar qué clientes probablemente recomprarán durante los próximos 30 días. RFM resuelve el problema secundario: resume el historial de cada persona en variables que el modelo posterior podrá usar, siempre que no mezclen pagos, órdenes y personas en un mismo grano.
</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 RFM</strong>
  <strong>RFM</strong> resume el comportamiento de compra de una persona en una fecha de referencia. <strong>Recency</strong> mide los días desde la última compra elegible; <strong>Frequency</strong> cuenta órdenes elegibles; y <strong>Monetary</strong> suma sus pagos. En este laboratorio la persona se identifica con `customer_unique_id`, solo cuentan órdenes entregadas con pago registrado y la fecha de referencia es el día posterior a la última compra elegible.
</div>

<img src="assets/rfm.png" alt="Diagrama de segmentación RFM de Pudubella que define Recency, Frequency y Monetary y muestra los segmentos campeones, potenciales, en riesgo e inactivos." style="max-width:100%; height:auto;">

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Cobertura y exclusiones</strong>
  La <strong>cobertura</strong> declara qué registros entran a un producto y cuáles quedan fuera. Una orden entregada sin pago no recibe un cero: queda en `gold.rfm_exclusions` con su motivo y no entra al RFM monetario. Conservar esa exclusión permite distinguir “no hubo compra” de “no hay evidencia suficiente para medirla”.
</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Variables predictivas y fuga de información</strong>
  Para estimar propensión a recompra, RFM funciona como un conjunto de <strong>variables predictivas</strong> y el resultado es si la persona recompra o no en los próximos 30 días. Las variables deben calcularse solo con información disponible antes de ese período. Usar una compra posterior a la fecha de referencia para construir RFM sería <strong>fuga de información</strong>: el modelo parecería acertar porque recibió parte de la respuesta. Por eso la limpieza, las exclusiones y la fecha de referencia no son detalles del pipeline; definen qué evidencia recibe el modelo.
</div>

### 3.1 — Cobertura antes de construir el RFM (0,4 puntos)

Una unión combina todas las coincidencias de sus claves. Si un préstamo tiene dos libros y dos cargos, unir ambas tablas directamente genera cuatro filas; sumar después duplica montos. Primero se agrega cada lado al grano que corresponde, y luego se une.

In [ ]:
prestamos = pl.DataFrame({"prestamo_id": ["p1", "p1"], "libro": ["A", "B"]})
cargos = pl.DataFrame({"prestamo_id": ["p1", "p1"], "monto": [1000, 500]})
prestamos.join(cargos, on="prestamo_id").select(
    pl.len().alias("filas_despues_del_join"),
    pl.col("monto").sum().alias("monto_incorrecto"),
)

Construyan `ordenes_sin_pago` desde las entidades Silver ya verificadas. Usen un
anti join entre órdenes entregadas y las claves distintas de pagos; conserven
`order_id`, `customer_id` y `order_purchase_timestamp`, y muestren el conteo.
Esta exploración define qué órdenes quedan fuera de RFM antes de implementar el
producto Gold.

In [ ]:
# Su código aquí
raise NotImplementedError(
    "Completen la auditoría de órdenes entregadas sin pago antes de continuar."
)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Preguntas</strong>
  1. ¿Qué representa una fila de `gold.sales_daily` y una de `gold.customer_rfm`?
  2. Antes de ejecutar el código de producción, ¿cuántas órdenes entregadas sin pago predicen que aparecerán en `ordenes_sin_pago`? ¿Qué persona y qué métrica deja fuera esa exclusión?
  3. ¿Por qué deben agregarse los pagos por `order_id` antes de unirlos al RFM?
  4. Los umbrales en `config/rfm_segments.yaml` fueron perfilados sobre este extracto. ¿Por qué no deben asumirse universales?
</div>

<code>Escriban sus respuestas aquí:</code>

### 3.2 — Construcción exploratoria de productos Gold (1,4 puntos)

Construyan los productos con las entidades Silver que verificaron en la etapa
anterior. La auditoría previa ya identificó las exclusiones de RFM: reutilicen
esa evidencia en lugar de volver a calcularla. Al final, trasladarán las tres
transformaciones, sus contratos y la configuración de segmentos al módulo
reutilizable.

#### 3.2.1 — Ventas diarias

Construyan `sales_daily`. Consideren solo órdenes `delivered`, únanlas con sus
ítems por `order_id` y creen `sale_date` desde la fecha de compra. El resultado
debe tener una fila por `sale_date`, `items_sold_value` como suma de `price` y
`delivered_orders` como número de `order_id` distintos. Los fletes no forman
parte de esta métrica.

In [ ]:
# Su código aquí
raise NotImplementedError(
    "Construyan sales_daily antes de trasladarlo a gold.py."
)

#### 3.2.2 — Exclusiones y variables RFM por persona

Conviertan primero `ordenes_sin_pago` en `rfm_exclusions`: una fila por
`order_id`, las columnas que ya conservaron y `reason` igual a
`delivered_order_without_payment`. Esa tabla no entra al RFM ni transforma un
pago faltante en cero.

Luego construyan `customer_rfm` solo desde órdenes entregadas con pago
registrado. Antes de unir pagos a clientes, agréguenlos por `order_id`; una
orden puede tener más de un pago y sumar después de un join al grano de cliente
duplicaría `Monetary`. Luego relacionen las órdenes con `customer_unique_id`,
agrupen por esa identidad y calculen:

- `last_purchase`: última fecha de compra elegible;
- `frequency`: número de órdenes elegibles distintas;
- `monetary`: suma de pagos de esas órdenes;
- `recency_days`: días calendario entre `last_purchase` y el día posterior a
  la última compra elegible del conjunto;
- `segment`: segmento calculado con `config/rfm_segments.yaml`.

No usen compras posteriores a la fecha de referencia ni incluyan una orden de
`rfm_exclusions`.

In [ ]:
# Su código aquí
raise NotImplementedError(
    "Construyan rfm_exclusions y customer_rfm antes de trasladarlos a gold.py."
)

<!-- MINI PROYECTO -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240,136,62,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🛠️ Modularización — Productos analíticos</strong>
  Trasladen las tres transformaciones a `src/pudulake/gold.py`: `build_sales_daily`, `build_rfm_exclusions` y `build_customer_rfm`. Reciben entidades Silver, devuelven una tabla y no escriben archivos. Conserven los mismos granos, filtros, exclusiones y fecha de referencia que verificaron en el notebook.

  Completen ahora `gold_sales_daily.yaml`, `gold_rfm_exclusions.yaml` y `gold_customer_rfm.yaml`. Cada contrato debe declarar los campos obligatorios, el grano, la llave, el lineage desde Silver, los tipos y las restricciones del producto. Usen los tipos y reglas permitidos por `contracts.py`. El flow validará estos contratos antes de publicar Gold.

  `gold.customer_rfm` es la tabla de variables para un modelo posterior de propensión a recompra; no entrenen ni evalúen ese modelo en este laboratorio.
</div>

In [ ]:
from src.pudulake.gold import (
    build_customer_rfm,
    build_rfm_exclusions,
    build_sales_daily,
)

# Después de modularizar, esta comprobación debe ejecutarse sin modificarla.
contracts_gold = {
    path.stem: load_contract(path)
    for path in sorted((RUTA_LAB / "contracts").glob("gold_*.yaml"))
}
sales_daily = build_sales_daily(orders_silver, items_silver)
rfm_exclusions = build_rfm_exclusions(orders_silver, payments_silver)
customer_rfm = build_customer_rfm(
    orders_silver,
    customers_silver,
    payments_silver,
    {"segments": segments},
)
validate_contract(sales_daily, contracts_gold["gold_sales_daily"])
validate_contract(rfm_exclusions, contracts_gold["gold_rfm_exclusions"])
validate_contract(customer_rfm, contracts_gold["gold_customer_rfm"])
pl.DataFrame(
    {
        "producto": ["sales_daily", "rfm_exclusions", "customer_rfm"],
        "filas": [
            sales_daily.height,
            rfm_exclusions.height,
            customer_rfm.height,
        ],
    }
)

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  `uv run pytest -m etapa3` pasa. `gold.sales_daily` tiene 4 filas, `gold.customer_rfm` tiene 2 y `gold.rfm_exclusions` conserva exactamente 1 orden: la misma que auditaron en `ordenes_sin_pago`.
</div>

---
## Etapa 4 — Orquestación, lineage y entrega (1,0 puntos)

<!-- LORE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #0f9e99; background:rgba(15,158,153,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📜 El rastro que falta</strong>
  Don Pudencio ve las tablas Gold, pero pregunta qué se rompería si cambia `silver.orders`. El equipo debe poder responder con evidencia del flow y no con un diagrama dibujado después.
</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Lineage</strong>
  <strong>Lineage</strong> es la evidencia de cómo se obtuvo una tabla: qué insumos usó, qué transformaciones la produjeron y qué productos dependen de ella. Si cambia `silver.orders`, el lineage permite identificar que deben revisarse `gold.sales_daily`, `gold.customer_rfm`, `gold.rfm_exclusions` y las variables que un modelo posterior reciba desde RFM.
</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63,185,80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Orquestación, assets y snapshots</strong>
  La <strong>orquestación</strong> decide el orden de las tareas, registra sus estados y detiene las dependencias cuando una tarea falla. Un <strong>asset</strong> representa una tabla o producto materializado; una <strong>materialización</strong> registra que ese asset se construyó en una corrida. Un <strong>snapshot</strong> conserva una versión del catálogo para poder inspeccionar qué tablas estaban publicadas en ese momento.
</div>

### 4.1 — Flow observable y dependencias (0,3 puntos)

Ejecuten primero el flow normal desde el notebook. La salida debe informar
`Ventas diarias: 4`, `RFM: 2` y `exclusiones RFM: 1`. Luego consulten el
estado del lakehouse y los snapshots. Conserven la salida: será la evidencia
para comparar el caso inválido.

In [ ]:
import subprocess

subprocess.run(
    ["uv", "run", "python", "main.py", "run"],
    cwd=RUTA_LAB,
    check=True,
)
subprocess.run(
    ["uv", "run", "python", "main.py", "status"],
    cwd=RUTA_LAB,
    check=True,
)
subprocess.run(
    ["uv", "run", "python", "main.py", "snapshots"],
    cwd=RUTA_LAB,
    check=True,
)

### 4.2 — El caso inválido (0,5 puntos)

<!-- WARNING -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #d29922; background:rgba(210,153,34,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">⚠️ El caso inválido</strong>
  `data/raw_invalid/payments.parquet` contiene un pago negativo. Ejecutar el flow con <code>--raw-dir data/raw_invalid</code> debe fallar antes de publicar Silver. Bronze puede conservar esa ingesta para trazabilidad; `status` debe identificar la falla y mantener visible cuándo se construyó el último Gold válido. No reemplacen los datos normales ni corrijan el archivo inválido.
</div>

Ejecuten el caso inválido después de la corrida válida. El comando debe terminar
con código de salida `2` y nombrar el contrato incumplido. Luego vuelvan a
consultar `status`: no basta con que el flow falle; deben verificar que el Gold
válido anterior sigue disponible.

In [ ]:
resultado_invalido = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "main.py",
        "run",
        "--raw-dir",
        "data/raw_invalid",
    ],
    cwd=RUTA_LAB,
    capture_output=True,
    text=True,
)
print(resultado_invalido.stderr)
if resultado_invalido.returncode != 2:
    raise RuntimeError("El caso inválido no terminó con el diagnóstico esperado.")
subprocess.run(
    ["uv", "run", "python", "main.py", "status"],
    cwd=RUTA_LAB,
    check=True,
)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88,166,255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Evidencia del flow</strong>
  1. Separen las once tablas de la corrida válida por capa y nombren el snapshot que registra Gold.
  2. ¿Qué regla detuvo la corrida inválida y qué evidencia de `status` muestra que el Gold anterior no fue reemplazado?
  3. Si cambia la semántica de `silver.orders`, ¿qué productos Gold deben revisarse y qué evidencia del grafo permite afirmarlo?
</div>

<code>Escriban su respuesta aquí:</code>

<!-- MINI PROYECTO -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240,136,62,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🛠️ Modularización — Flow observable</strong>
  En esta etapa la modularización del flow ya está provista en `src/pudulake/flows.py`, `src/pudulake/storage.py` y `main.py`; no modifiquen esa infraestructura. Su trabajo es comprobar que los módulos que implementaron antes se integran en una corrida completa: Bronze conserva las fuentes, Silver se publica como conjunto tras validar sus relaciones y Gold solo cambia cuando todas sus dependencias son válidas.

  Entreguen una captura del flow exitoso y otra del grafo de assets en Prefect Server. La evidencia debe permitir seguir el lineage desde `silver.orders` hacia los tres productos Gold.
</div>

<!-- COMPROBACIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #2da44e; background:rgba(45,164,78,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🧪 Comprobación</strong>
  `uv run pytest -m etapa4` pasa. Después de la corrida válida, `uv run python main.py status` lista once tablas y `uv run python main.py snapshots` muestra snapshots. La ejecución con los datos inválidos termina con código `2`, registra el contrato incumplido y conserva el último Gold válido.
</div>

### Entrega (0,2 puntos)

- [ ] Pull Request con los módulos, contratos y notebook completos.
- [ ] Todas las pruebas pasan.
- [ ] Captura del flow exitoso y del grafo de assets.
- [ ] Respuestas de semántica, grano e impacto completadas.

Las dudas se reciben en el foro del curso o por los canales habituales antes del plazo de entrega.